# 06b — Routine Representation Robustness

Stage 06 found recurring OD structure, but the representation still needs robustness checks before change detection.

```text
23 users with OD repeated on ≥2 days
9 users on ≥3 days
2 users on ≥5 days

same dominant OD across halves ≈ 13.3%
```

06b asks whether this is real temporal instability or mostly sparse-sampling rank swaps.

## Design

06b adds four controls:

```text
1. support-aware collapsed-sequence motif comparator
2. full OD-distribution split-half metrics
3. chronological split vs random balanced day partitions
4. day-level bootstrap for departure-time concentration
5. stricter multimodal GMM validation
```

No HOME/WORK semantics are used.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
CACHE_06 = VOLUME_ROOT / "cache" / "06_routine_habit_mining"
CACHE_06B = VOLUME_ROOT / "cache" / "06b_routine_representation_robustness"
CACHE_06B.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

if not (REPO_DIR / ".git").exists():
    run([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
        "https://github.com/tanh1c/geolife.git", str(REPO_DIR)
    ])
else:
    current = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "branch", "--show-current"],
        text=True,
    ).strip()
    if current != REPO_BRANCH:
        run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])

for candidate in (REPO_DIR, REPO_DIR / "src"):
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

spec = importlib.util.spec_from_file_location(
    "stage06b",
    REPO_DIR / "analysis" / "06b_routine_representation_robustness.py",
)
stage06b = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = stage06b
spec.loader.exec_module(stage06b)

display(pd.DataFrame([stage06b.synthetic_self_check()]))
print("Ready:", CACHE_06B)

## 1. Reuse Stage-06 caches

06b does not rerun CP1, timezone resolution, or complete-link clustering.

It consumes the already executed Stage-06 private artifacts.

In [ ]:
paths = {
    "day_sequences": CACHE_06 / "day_sequences_private.pkl",
    "transitions": CACHE_06 / "transitions_private.pkl",
    "edge_summary": CACHE_06 / "edge_summary_private.pkl",
    "user_summary": CACHE_06 / "user_summary_private.pkl",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebook 06 first:\n" + "\n".join(missing))

day_sequences = pd.read_pickle(paths["day_sequences"])
transitions = pd.read_pickle(paths["transitions"])
edge_summary = pd.read_pickle(paths["edge_summary"])
user_summary = pd.read_pickle(paths["user_summary"])

display(pd.DataFrame([{
    "supported_users": day_sequences["user_id"].nunique(),
    "supported_days": len(day_sequences),
    "transitions": len(transitions),
    "distinct_user_edges": len(edge_summary),
}]))

## 2. Support-aware motif comparator

Primary comparison uses the same observation universe:

```text
usable motif days ≥ 6

repeated OD:
same directed edge on ≥3 days

supported collapsed-sequence motif:
top motif on ≥3 days
AND ≥50% of usable days
```

This fixes the Stage-06 issue where one observed day could appear as 100% repeatable.

In [ ]:
audit06b = stage06b.run_audit(
    day_sequences,
    transitions,
    edge_summary,
)

display(audit06b.motif_primary)
display(audit06b.motif_sensitivity)
display(audit06b.support_tiers)

## 3. Distributional split-half stability

Exact top-edge equality is too brittle.

06b reports:

```text
JSD                 0 = identical
weighted Jaccard    1 = identical
total variation     0 = identical
top-3 Jaccard       1 = identical
```

In [ ]:
display(audit06b.split_aggregate)

split = audit06b.split_distribution.loc[
    audit06b.split_distribution["comparable"].fillna(False).astype(bool)
]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(split["jsd"].dropna(), bins=12)
ax.set(
    xlabel="chronological split JSD",
    ylabel="users",
    title="First-half vs second-half OD distribution difference",
)
plt.show()

## 4. Chronological difference vs sampling-only partitions

For each comparable user:

```text
chronological first/second half
vs
200 random balanced day partitions
```

If chronological JSD exceeds random-partition p95, temporal ordering contains more drift-like structure than expected from support variability alone.

In [ ]:
random = audit06b.random_partition

random_summary = pd.DataFrame([{
    "calibrated_users": random["user_id"].nunique(),
    "median_chronological_jsd": float(random["chronological_jsd"].median()),
    "median_random_jsd": float(random["random_jsd_median"].median()),
    "median_chrono_minus_random_jsd": float(
        random["chronological_minus_random_median_jsd"].median()
    ),
    "users_above_random_p95": int(
        random["chronological_above_random_p95_jsd"].sum()
    ),
    "share_above_random_p95": float(
        random["chronological_above_random_p95_jsd"].mean()
    ),
}])
display(random_summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(random["random_jsd_median"], random["chronological_jsd"])
ax.plot([0, 1], [0, 1])
ax.set(
    xlim=(0, 1), ylim=(0, 1),
    xlabel="median random-partition JSD",
    ylabel="chronological JSD",
    title="Temporal difference vs sampling-only partitions",
)
plt.show()

## 5. Bootstrap departure-time regularity

One circular-mean departure hour is built per edge-day.

Active days are bootstrapped, so a high concentration estimated from very few days is visibly uncertain.

In [ ]:
display(audit06b.departure_bootstrap_summary)

boot = audit06b.departure_bootstrap
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(boot["active_days"], boot["ci95_low"])
ax.set(
    xlabel="active days",
    ylabel="95% lower bound of departure concentration",
    title="Clock regularity evidence vs OD support",
)
plt.show()

## 6. Stricter multimodal departure-time evidence

Primary strict gate:

```text
≥5 active days
≥8 transitions
BIC gain vs 1 mode ≥10
component weight ≥0.20
mode centers separated by ≥2h
```

These are robustness filters, not semantic labels.

In [ ]:
strict = audit06b.strict_modes
display(pd.DataFrame([{
    "modeled_edges": len(strict),
    "modeled_users": strict["user_id"].nunique() if len(strict) else 0,
    "strict_multimodal_edges": int(strict["strict_multimodal"].sum()) if len(strict) else 0,
    "strict_multimodal_users": (
        strict.loc[strict["strict_multimodal"], "user_id"].nunique()
        if len(strict) else 0
    ),
}]))

display(audit06b.strict_mode_sensitivity)

## 7. Stage-07 readiness snapshot

No synthetic readiness score is created.

The table keeps support, distributional stability, sampling calibration, departure-time robustness, and multimodality as separate evidence axes.

In [ ]:
display(audit06b.readiness)

# Private outputs.
audit06b.split_distribution.to_pickle(CACHE_06B / "split_distribution_private.pkl")
audit06b.random_partition.to_pickle(CACHE_06B / "random_partition_private.pkl")
audit06b.departure_bootstrap.to_pickle(CACHE_06B / "departure_bootstrap_private.pkl")
audit06b.strict_modes.to_pickle(CACHE_06B / "strict_modes_private.pkl")

# Aggregate outputs.
audit06b.motif_primary.to_csv(CACHE_06B / "motif_primary.csv", index=False)
audit06b.motif_sensitivity.to_csv(CACHE_06B / "motif_sensitivity.csv", index=False)
audit06b.support_tiers.to_csv(CACHE_06B / "support_tiers.csv", index=False)
audit06b.split_aggregate.to_csv(CACHE_06B / "split_aggregate.csv", index=False)
random_summary.to_csv(CACHE_06B / "random_partition_summary.csv", index=False)
audit06b.departure_bootstrap_summary.to_csv(
    CACHE_06B / "departure_bootstrap_summary.csv", index=False
)
audit06b.strict_mode_sensitivity.to_csv(
    CACHE_06B / "strict_mode_sensitivity.csv", index=False
)
audit06b.readiness.to_csv(CACHE_06B / "readiness.csv", index=False)

print("Saved:", CACHE_06B)

## Decision gate

Proceed to Stage 07 only if the executed results show that:

```text
support retains a non-trivial cohort
+
distributional stability is interpretable
+
chronological drift can be separated from random partitions
+
clock regularity strengthens with support
+
multimodality survives stricter checks
```

If low top-1 stability disappears under distributional metrics, Stage 07 should operate on edge distributions rather than top-edge identity.